# Спайк 2 — LiveKit turn-detector multilingual (текстовая модель)

Семантический детектор конца реплики по **транскрипту** (от ASR).

- Базовая модель: Qwen2.5-0.5B-Instruct, дистиллирована, ONNX INT8
- Вход: история диалога (до 6 ходов) в формате Qwen chat template,
  у последней реплики пользователя убран закрывающий `<|im_end|>`
- Выход: вероятность того, что дальше идёт `<|im_end|>` = реплика завершена
- 14 языков включая русский, для каждого свой порог в `languages.json`
- HF: https://huggingface.co/livekit/turn-detector

ВАЖНО: мультиязычная модель лежит **не в main**, а в ревизии `v0.4.1-intl`
(файл `onnx/model_q8.onnx` + `languages.json`). В main лежит старая English
SmolLM2-версия — она работает иначе.

## Установка зависимостей

In [1]:
# !pip install onnxruntime transformers huggingface_hub jinja2 numpy

## Загрузка модели (ревизия v0.4.1-intl)

In [2]:
import json
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

REPO = "livekit/turn-detector"
REVISION = "v0.4.1-intl"          # мультиязычная модель

tokenizer = AutoTokenizer.from_pretrained(REPO, revision=REVISION)
onnx_path = hf_hub_download(REPO, "onnx/model_q8.onnx", revision=REVISION)
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# пороги по языкам (подобраны авторами под высокий recall)
languages = json.load(open(hf_hub_download(REPO, "languages.json", revision=REVISION)))
RU_THRESHOLD = languages["ru"]["threshold"]

print("output:", [(o.name, o.shape) for o in session.get_outputs()])
print("ru threshold:", RU_THRESHOLD, "| meta:", languages["ru"])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


output: [('prob', ['batch_size', 'sequence_length'])]
ru threshold: 0.0032 | meta: {'threshold': 0.0032, 'tpr': 0.9933, 'tnr': 0.8803}


## Функция предсказания

Модель отдаёт вероятность EOU для каждой позиции — берём последнюю.
ONNX уже содержит softmax, дополнительной нормализации не нужно.

In [3]:
def eou_prob(text: str, context: list[dict] | None = None) -> float:
    """Вероятность что реплика пользователя завершена (0..1).

    context — предыдущие ходы диалога, напр.
        [{"role": "assistant", "content": "..."},
         {"role": "user", "content": "..."}]
    """
    messages = (context or []) + [{"role": "user", "content": text}]
    s = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    s = s.rsplit("<|im_end|>", 1)[0]      # убираем закрывающий тег у последней реплики
    ids = tokenizer(s, return_tensors="np", add_special_tokens=False)["input_ids"].astype(np.int64)
    prob = session.run(None, {"input_ids": ids})[0]   # [1, seq]
    return float(prob[0, -1])

## Тест на русских фразах

In [4]:
tests = [
    ("Здравствуйте, я хотел бы узнать баланс по своей карте", "complete"),
    ("Мне нужно",                                            "incomplete"),
    ("Скажите пожалуйста, какой у меня",                     "incomplete"),
    ("Я хочу оплатить счёт за электричество",                "complete"),
    ("Мой номер телефона восемь девятьсот",                  "incomplete"),
    ("Спасибо, до свидания",                                 "complete"),
    ("А можно ли",                                           "incomplete"),
    ("Какие у вас есть тарифы?",                             "complete"),
]

print(f"ru threshold = {RU_THRESHOLD}")
print(f"{'expected':<11}{'score':>8} {'pred':>11}  text")
for text, exp in tests:
    p = eou_prob(text)
    pred = "complete" if p > RU_THRESHOLD else "incomplete"
    mark = "OK" if pred == exp else "XX"
    print(f"{exp:<11}{p:>8.4f} {pred:>11} {mark}  {text}")

ru threshold = 0.0032
expected      score        pred  text
complete     0.4005    complete OK  Здравствуйте, я хотел бы узнать баланс по своей карте
incomplete   0.0070    complete XX  Мне нужно
incomplete   0.0038    complete XX  Скажите пожалуйста, какой у меня
complete     0.1764    complete OK  Я хочу оплатить счёт за электричество
incomplete   0.1221    complete XX  Мой номер телефона восемь девятьсот
complete     0.9444    complete OK  Спасибо, до свидания
incomplete   0.0003  incomplete OK  А можно ли
complete     0.6343    complete OK  Какие у вас есть тарифы?


## Замечание про порог

Дефолтный `ru threshold = 0.0032` подобран авторами под **замену VAD** —
максимальный recall (TPR ~0.99), поэтому почти всё классифицируется как
`complete`. Для нашей формулы важен не бинарный вердикт модели, а сам
**score** — мы сравниваем его со своим порогом (например 0.9).

Score упорядочен правильно: `"А можно ли"` → ~0.0003, `"Спасибо, до свидания"` → ~0.94.

## Эффект контекста диалога

In [5]:
context = [
    {"role": "assistant", "content": "Назовите номер вашей карты"},
]
phrase = "Пять тысяч двести"
print("без контекста:", round(eou_prob(phrase), 4))
print("с контекстом :", round(eou_prob(phrase, context), 4))

без контекста: 0.2576
с контекстом : 0.7055


## Замер скорости инференса

In [6]:
import time
eou_prob("прогрев")
t0 = time.perf_counter()
N = 50
for _ in range(N):
    eou_prob("Я хочу оплатить счёт за электричество")
print(f"avg latency: {(time.perf_counter()-t0)/N*1000:.1f} ms / вызов")

avg latency: 9.9 ms / вызов


## Как встроить в наш пайплайн

```
если (eou_prob(accumulated_text) > 0.9 И тишина > 1с)
ИЛИ тишина > 5с
→ отправляем в LLM
```

Зовём на накопленном транскрипте от Qwen ASR. Минус — ждём транскрипт
(добавляет задержку), плюс — понимает смысл.

### Следующий шаг
Прогнать на транскриптах реальных звонков, сравнить со Smart Turn по
FP/FN при нашем пороге (см. `docs/05_metrics.md`).